# 1 - Introduction

Welcome to Playground Season 3 Episode 8, return of the RMSE! This time we're looking at [Gemstone Price Predictions](https://www.kaggle.com/datasets/colearninglounge/gemstone-price-prediction), so let's dive right in.

# 1.1 - Initial Dataset Impressions

First, let's examine some of the memory features of the datasets that we're looking at.

| Dataset | Size on Disk | Size in Memory |
| ------- | ------------ | -------------- |
| `train` | 10 MB        | 16.25 MB       |
| `test`  | 6.2 MB       | 9.85 MB        |

As we can see, memory pressure isn't too bad, nor is disk space storage. This means that we won't likely have too many problems with various machine learning features. Let's just take a look briefly at the training and testing sets.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc

import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv("../input/playground-series-s3e8/train.csv")
test = pd.read_csv("../input/playground-series-s3e8/test.csv")

train

In [ ]:
test

So this time around, we have lots of training data, and lots of testing data. Relatively fewer features to work with, but a mix of both categorical and continuous.

### Key Observations About Initial Impressions

* Size on disk and in memory is low, meaning that memory pressure is unlikely to be an issue for the competition.
* There are a total of 9 features that we can use for training, and one to predict - the price.
* In terms of sample sizes:
    * There are 193,573 rows to train on
    * There are 129,050 rows to test on

# 1.2 - Null Values

Let's explore the issue of missing values in the dataset to see if there are systemic problems with data representation.

In [ ]:
train["null_count"] = train.isnull().sum(axis=1)
counts = train.groupby("null_count")["id"].count().to_dict()
null_data = {"{} Null Value(s)".format(k) : v for k, v in counts.items() if k < 8}

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 7))

axs = axs.flatten()

_ = axs[0].pie(
    x=list(null_data.values()), 
    autopct=lambda x: "{:,.0f} = {:.2f}%".format(x * sum(null_data.values())/100, x),
    explode=[0.05] * len(null_data.keys()), 
    labels=null_data.keys(), 
    colors=sns.color_palette("Set2")[0:3],
)
_ = axs[0].set_title("Number of Null Values Per Row (Train)", fontsize=15)

test["null_count"] = test.isnull().sum(axis=1)
counts = test.groupby("null_count")["id"].count().to_dict()
null_data = {"{} Null Value(s)".format(k) : v for k, v in counts.items() if k < 8}

_ = axs[1].pie(
    x=list(null_data.values()), 
    autopct=lambda x: "{:,.0f} = {:.2f}%".format(x * sum(null_data.values())/100, x),
    explode=[0.05] * len(null_data.keys()), 
    labels=null_data.keys(), 
    colors=sns.color_palette("Set1")[0:3],
)
_ = axs[1].set_title("Number of Null Values Per Row (Test)", fontsize=15)

train = train.drop("null_count", axis=1)
test = test.drop("null_count", axis=1)

As in previous episodes, we don't see any null values appearing in our data. This is good. The other thing we should look for are zero values as well. This is based on observations in discussion threads [here](https://www.kaggle.com/competitions/playground-series-s3e8/discussion/389266). Let's check for missing values of `x`, `y`, and `z`.

In [ ]:
train[(train["y"] == 0) | (train["x"] == 0) | (train["z"] == 0)]

There are a total of 10 rows that have at least one of `x`, `y`, or `z` missing. There is a relationship between these numbers however, and the `depth` field. Assuming we have 2 out of the three values, we can compute the missing value with a simple formula from [here](https://www.kaggle.com/competitions/playground-series-s3e8/discussion/389266). In the cases above, we see we're really only missing `z`. We can impute that number fairly simply with the following function:

In [ ]:
train["has_missing_xyz"] = train[["x", "y", "z"]].apply(lambda row: 1 if row.x == 0 or row.y == 0 or row.z == 0 else 0, axis=1)

def impute_missing_xyz(df):
    df["z"] = df[["x", "y", "z", "depth"]].apply(
        lambda row: row.z if row.z > 0 else 0 if row.x == 0 or row.y == 0 else (row.depth * (row.x + row.y)) / 200.
    , axis=1)
    return df

When we apply the function, we can see imputed values. Notice however, we can't do anything where two or more columns are missing.

In [ ]:
train_copy = train.copy()
train_copy = impute_missing_xyz(train_copy)
train_copy[(train_copy["has_missing_xyz"] == 1)]

### Key Observations About Null Values

* No null values appear in the training or testing datasets.
* There are examples where the `x`, `y`, and `z` values are missing:
    * In train there are 10 rows.
    * In test there are 14 rows.

# 1.3 - Train / Test Difference - Adversarial Validation

As a quick test, we should see how different the values are between train and test. To do so, we'll quickly perform a round of adversarial validation to see if a classifier can tell the two datasets apart. We'll use ROC AUC score to inform us of differences. If the two sets appear very similar, the classifier will not be able to tell them apart, and thus will have an ROC AUC score of 0.5. If they are easy to tell apart - and thus are dissimilar - then the ROC AUC score will approach 1.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from lightgbm import early_stopping, log_evaluation
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.preprocessing import LabelEncoder

train["origin"] = 0
test["origin"] = 1

combined = train.copy()
combined = pd.concat([combined, test]).reset_index(drop=True)

features = [
    'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z'
]

for feature in ["cut", "color", "clarity"]:
    le = LabelEncoder()
    combined[feature] = le.fit_transform(combined[feature])
    
n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.zeros((combined.shape[0],))
train_oof_probas = np.zeros((combined.shape[0],))

for fold, (train_index, test_index) in enumerate(skf.split(combined, combined["origin"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(combined.iloc[train_index]), pd.DataFrame(combined.iloc[test_index])
    y_train, y_valid = combined["origin"].iloc[train_index], combined["origin"].iloc[test_index]
    
    x_train_features = pd.DataFrame(x_train[features])
    x_valid_features = pd.DataFrame(x_valid[features])

    model = LGBMClassifier(
        random_state=2023,
        objective="binary",
        metric="auc",
        n_jobs=-1,
        n_estimators=2000,
        verbose=-1,  
        max_depth=3,
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(2000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    oof_probas = model.predict_proba(x_valid_features[features])[:,1]
    train_oof_preds[test_index] = oof_preds
    train_oof_probas[test_index] = oof_probas
    print(": AUC ROC = {}".format(roc_auc_score(y_valid, oof_probas)))
    
auc_vanilla = roc_auc_score(combined["origin"], train_oof_probas)
fpr, tpr, _ = roc_curve(combined["origin"], train_oof_probas)
print("--> Overall results for out of fold predictions")
print(": AUC ROC = {}".format(auc_vanilla))

confusion = confusion_matrix(combined["origin"], train_oof_preds)

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15, 8))

_ = sns.heatmap(confusion, annot=True, fmt=",d", ax=axs[0])
_ = axs[0].set_title("Confusion Matrix (@ 0.5 Probability)", fontsize=15)
_ = axs[0].set_ylabel("Actual Class")
_ = axs[0].set_xlabel("Predicted Class")

_ = sns.lineplot(x=[0, 1], y=[0, 1], linestyle="--", label="Indistinguishable Datasets", ax=axs[1])
_ = sns.lineplot(x=fpr, y=tpr, ax=axs[1], label="Adversarial Validation Classifier")
_ = axs[1].set_title("ROC Curve", fontsize=15)
_ = axs[1].set_xlabel("False Positive Rate")
_ = axs[1].set_ylabel("True Positive Rate")

### Key Observations About Train / Test Difference - Adversarial Validation

* The trained classifier has an ROC AUC score of 0.5028, which suggests that the training dataset and the testing dataset are very similar.

# 1.4 - Original Data - Adversarial Validation

One thing we can also look at is the original dataset, in this case the [Gemstone Price Predictions](https://www.kaggle.com/datasets/colearninglounge/gemstone-price-prediction) dataset. We will perform an adversarial validation to see whether that dataset is similar to the competition dataset.

In [ ]:
original = pd.read_csv("../input/gemstone-price-prediction/cubic_zirconia.csv")

train["origin"] = 0
test["origin"] = 0
original["origin"] = 1

combined = train.copy()
combined = pd.concat([combined, original]).reset_index(drop=True)

features = [
    'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z'
]

for feature in ["cut", "color", "clarity"]:
    le = LabelEncoder()
    combined[feature] = le.fit_transform(combined[feature])
    
n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.zeros((combined.shape[0],))
train_oof_probas = np.zeros((combined.shape[0],))

for fold, (train_index, test_index) in enumerate(skf.split(combined, combined["origin"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(combined.iloc[train_index]), pd.DataFrame(combined.iloc[test_index])
    y_train, y_valid = combined["origin"].iloc[train_index], combined["origin"].iloc[test_index]
    
    x_train_features = pd.DataFrame(x_train[features])
    x_valid_features = pd.DataFrame(x_valid[features])

    model = LGBMClassifier(
        random_state=2023,
        objective="binary",
        metric="auc",
        n_jobs=-1,
        n_estimators=2000,
        verbose=-1,  
        max_depth=3,
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(2000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    oof_probas = model.predict_proba(x_valid_features[features])[:,1]
    train_oof_preds[test_index] = oof_preds
    train_oof_probas[test_index] = oof_probas
    print(": AUC ROC = {}".format(roc_auc_score(y_valid, oof_probas)))
    
auc_vanilla = roc_auc_score(combined["origin"], train_oof_probas)
fpr, tpr, _ = roc_curve(combined["origin"], train_oof_probas)
print("--> Overall results for out of fold predictions")
print(": AUC ROC = {}".format(auc_vanilla))

confusion = confusion_matrix(combined["origin"], train_oof_preds)

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15, 8))

_ = sns.heatmap(confusion, annot=True, fmt=",d", ax=axs[0])
_ = axs[0].set_title("Confusion Matrix (@ 0.5 Probability)", fontsize=15)
_ = axs[0].set_ylabel("Actual Class")
_ = axs[0].set_xlabel("Predicted Class")

_ = sns.lineplot(x=[0, 1], y=[0, 1], linestyle="--", label="Indistinguishable Datasets", ax=axs[1])
_ = sns.lineplot(x=fpr, y=tpr, ax=axs[1], label="Adversarial Validation Classifier")
_ = axs[1].set_title("ROC Curve", fontsize=15)
_ = axs[1].set_xlabel("False Positive Rate")
_ = axs[1].set_ylabel("True Positive Rate")

We should also take a look to see if the original data has null or missing values.

In [ ]:
original["null_count"] = original.isnull().sum(axis=1)
counts = original.groupby("null_count")["Unnamed: 0"].count().to_dict()
null_data = {"{} Null Value(s)".format(k) : v for k, v in counts.items() if k < 8}

fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(5, 7))

_ = axs.pie(
    x=list(null_data.values()), 
    autopct=lambda x: "{:,.0f} = {:.2f}%".format(x * sum(null_data.values())/100, x),
    explode=[0.05] * len(null_data.keys()), 
    labels=null_data.keys(), 
    colors=sns.color_palette("Set2")[0:3],
)
_ = axs.set_title("Number of Null Values Per Row (Original)", fontsize=15)

It appears that there is null data in the original dataset. We probably want to either drop or fix the missing data. Let's look at the rows with null counts.

In [ ]:
original[(original["null_count"] > 0)]

From here it looks like the null data refers to the `depth` field. We can verify that by checking for the rows with a null depth value.

In [ ]:
original[(original["depth"].isnull())]

Indeed, we see that there are 697 rows that have null values for `depth`. Luckily, we can easily impute the missing values using a simple calculation as follows:

In [ ]:
def fix_depth(df):
    df["depth"] = df[["depth", "x", "y", "z"]].apply(
        lambda row: row.depth if row.depth == np.nan else ((row.z) / (row.x + row.y)) * 200, axis=1
    )
    return df

We should also check if there are zero values for x, y, or z.

In [ ]:
original[(original["x"] == 0) | (original["y"] == 0) | (original["z"] == 0)]

Finally, we should check for duplicate data in the original dataset.

In [ ]:
original["duplicated"] = original.duplicated(subset=["carat", "cut", "color", "clarity", "depth", "table", "x", "y", "z"])
original[(original["duplicated"] == True)]

### Key Observations About Original Data - Adversarial Validation

* The classifier has an AUC ROC score of 0.6313 - this means that the datasets are fairly similar to one another - moreso than what we have seen in previous entries in this Playground Season. It is likely we can mix these two datasets together, however we should still do some exploration to make sure they are sufficiently similar.
* The original data contains null values that need to be imputed.
* The original data has some rows that are missing x, y, or z values that need to be imputed.
* The original data has 88 duplicate rows.

# 1.5 - Statistical Breakdown

Let's take a closer look at some of the statistical properties of the continuous features. As a reminder, what we're looking for between the two sets is big differences between the min, max, and standard deviations for each continuous column. Differences there tell us if there are outliers that we need to deal with. Let's start with the training set.

In [ ]:
cont_features = [
    'depth', 'table', 'x', 'y', 'z'
]

train[cont_features].describe().T.style.bar(subset=['mean'], color='#7BCC70')\
    .background_gradient(subset=['std'], cmap='Reds')\
    .background_gradient(subset=['50%'], cmap='coolwarm')

And next at the test set.

In [ ]:
test[cont_features].describe().T.style.bar(subset=['mean'], color='#7BCC70')\
    .background_gradient(subset=['std'], cmap='Reds')\
    .background_gradient(subset=['50%'], cmap='coolwarm')

### Key Observations About Statistical Breakdown

* The mean, min, max, and standard deviations between train and test are remarkably similar, meaning that the two datasets share very similar properties.
    * This was confirmed by the adversarial validation above.

# 1.6 - Spearman Correlation

We should check to see if there is high correlation between our features. Spearman correlation does not make assumptions about distribution types or linearity. With Spearman correlation, we have values that range from -1 to +1. Values around either extreme end mean a neagative or positive correlation respectively, while those around 0 mean no correlation exists.

In [ ]:
features = [
    'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z', 'price'
]

correlation_matrix = train[features].corr(method="spearman")

from matplotlib.colors import SymLogNorm

f, ax = plt.subplots(figsize=(20, 20))
_ = sns.heatmap(
    correlation_matrix, 
    mask=np.triu(np.ones_like(correlation_matrix, dtype=bool)), 
    cmap=sns.diverging_palette(230, 20, as_cmap=True), 
    center=0,
    square=True, 
    linewidths=.1, 
    cbar=False,
    ax=ax,
    annot=True,
)
_ = ax.set_title("Spearman Correlation Matrix", fontsize=15)

### Key Observations About Spearman Correlation

* Unsurprisingly, the `carat` feature shows very strong correlations to `x`, `y`, and `z` - this is expected, as the carat rating of the gemstone is directly proportional to how large it is. 
* Similarly, the `price` of the gemstone is strongly correlated with `carat`, `x`, `y`, and `z`.

# 1.7 - P-Value Testing

While looking at features visually will tell us some interesting information, we can also use p-value testing to see if a feature has a net impact on a simple regression model. This method is controversial in that it likely doesn't provide a correct look at what features are informative. Our null hypothesis is that the feature impacts the target variable of `price`. In this case, anything with a p-value greater than 0.05 means we reject that hypothesis, and can potentially flag it for removal. Based on our current examination thus far, and given how few features we have to work with, it is very likely that all our features are informative.

In [ ]:
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

features = [
    'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z'
]

train_copy = train.copy()

for feature in ["carat", "cut", "color", "clarity"]:
    le = LabelEncoder()
    train_copy[feature] = le.fit_transform(train_copy[feature])
    
x = add_constant(train_copy[features])
model = OLS(train_copy["price"], x).fit()

pvalues = pd.DataFrame(model.pvalues)
pvalues.reset_index(inplace=True)
pvalues.rename(columns={0: "pvalue", "index": "feature"}, inplace=True)
pvalues.style.background_gradient(cmap='YlOrRd')

### Key Observations About P-Value Testing

* None of the features come up as candidates for removal, based on P-Value testing.

# 2 - Feature Exploration

Let's take a closer look at some of the features we have in the dataset.

# 2.1 - Clarity

According to the [American Gem Society](https://www.americangemsociety.org/buying-diamonds-with-confidence/ags-diamond-grading-system/), the clarity of a diamond corresponds to whether or not there are carbon inclusions embedded in the diamond itself. The scale actually has an ordering to it, from best to worst is:

* `IF` - Internally Flawless
* `VVS1` - Very Very Slightly Included 1
* `VVS2` - Very Very Slightly Included 2
* `VS1` - Very Slightly Included 1
* `VS2` - Very Slightly Included 2
* `SI1` - Slightly Included 1
* `SI2` - Slightly Included 2
* `I1` - Included 1

The clarity of the diamond can actually be expressed on a standard scale from 0 to 10, with 0 being the best possible score, and 10 being the worst possible score. The clarity we have expressed in the training and testing sets is a string value. Let's just take a look at it for a moment to see how it works out.

In [ ]:
def clarity_scale(df):
    df["clarity_scaled"] = df["clarity"].apply(
        lambda x: 0 if x == "IF" else 1 if x == "VVS1" else 2 if x == "VVS2" else 3 if x == "VS1" else 4 if x == "VS2" else 5 if x == "SI1" else 6 if x == "SI2" else 7
    )
    df["clarity_scaled"] = df["clarity_scaled"].astype(np.int8)
    return df

train = clarity_scale(train)

f, ax = plt.subplots(figsize=(15, 30))

_ = sns.violinplot(data=train, x="price", y="clarity", ax=ax)
_ = ax.set_title("Price Distributions by Clarity", fontsize=15)
_ = ax.set_xlabel("Price")
_ = ax.set_ylabel("Clarity")

### Key Observations About Clarity

* Although we would expect IF clarity gems to be the highest prices, we see the that they aren't most of the time. 
    * Arguably, both I1 and SI2 - some of the lowest clarity ratings - both have the bulk of their sales prices at a higher price point than IF or VVS clarity gems. 
* We can see that there are long tails for each clarity type. 
* While we were hoping for unique price points for each clarity type, the ordering of the clarity scale does give us a potential engineered feature.

# 2.2 - X and Y Values

The X and Y values of the diamond correspond to it's length and width. This is a nice series of measurements, since it can be plotted in a two dimensional space easily. Let's see how price factors in.

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 10))

_ = sns.scatterplot(data=train, x="x", y="y", hue="price", ax=ax)
_ = sns.lineplot(x=[0, 10], y=[0, 10], linestyle="--", label="X = Y", ax=ax)
_ = ax.set_title("X and Y vs Price", fontsize=15)
_ = ax.set_ylabel("Y")
_ = ax.set_xlabel("X")

As we can see, as the X and Y values increase, so does the overall price, in nearly a linear fashion. Also noteworthy, is the fact that most of the gem sizes appear to have a very tight series of aspect ratios. This is something we can look at a little closer with a kernel density estimate.

In [ ]:
train_copy = train.copy()
train_copy["y"] = train_copy["y"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["x"] = train_copy["x"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["aspect_ratio"] = train_copy["x"] / train_copy["y"]

fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(10, 10))

_ = sns.kdeplot(train_copy["aspect_ratio"], shade=True, color="r", ax=axs, label="Aspect Ratio Densities")
_ = axs.set_title("Aspect Ratio Densities", fontsize=15)
_ = axs.set_ylabel("Density")
_ = axs.set_xlabel("Aspect Ratio")
_ = axs.set_xlim((0.95, 1.05))

On the face of it, it looks like we are dealing with two different aspect ratios. However, is this really the case? The X and Y lengths are interpreted based on how you hold the gem. Turning the gem by 90 degrees flips the X and Y. So in terms of aspect ratios, it's not surprising that we see two distinct groups. If we instead flip the gem such that X is always the larger dimension, we should see a single concentration of densities. Let's do that and see what happens.

In [ ]:
train_copy = train.copy()
train_copy["y"] = train_copy["y"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["x"] = train_copy["x"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["new_y"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y <= row.x else row.x, axis=1)
train_copy["new_x"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y >= row.x else row.x, axis=1)
train_copy["aspect_ratio"] = train_copy["new_x"] / train_copy["new_y"]

fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(10, 10))

_ = sns.kdeplot(train_copy["aspect_ratio"], shade=True, color="r", ax=axs, label="Aspect Ratio Densities")
_ = axs.set_title("Aspect Ratio Densities", fontsize=15)
_ = axs.set_ylabel("Density")
_ = axs.set_xlabel("Aspect Ratio")
_ = axs.set_xlim((0.95, 1.3))

We can then see if there is any difference in prices.

In [ ]:
train_copy = train.copy()
train_copy["y"] = train_copy["y"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["x"] = train_copy["x"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["new_y"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y <= row.x else row.x, axis=1)
train_copy["new_x"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y >= row.x else row.x, axis=1)
train_copy["aspect_ratio"] = train_copy["new_x"] / train_copy["new_y"]

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 10))

_ = sns.scatterplot(data=train_copy, x="new_x", y="new_y", hue="price", ax=ax)
_ = sns.lineplot(x=[0, 10], y=[0, 10], linestyle="--", label="X = Y", ax=ax)
_ = ax.set_title("X and Y vs Price", fontsize=15)
_ = ax.set_ylabel("Y")
_ = ax.set_xlabel("X")

We've shifted the bulk of the aspect ratios to one side of the identity line. Let's see now if we can use aspect ratio to separate prices.

In [ ]:
train_copy = train.copy()
train_copy["y"] = train_copy["y"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["x"] = train_copy["x"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["new_y"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y <= row.x else row.x, axis=1)
train_copy["new_x"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y >= row.x else row.x, axis=1)
train_copy["aspect_ratio"] = train_copy["new_x"] / train_copy["new_y"]

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 10))

_ = sns.scatterplot(data=train_copy, x="aspect_ratio", y="price", hue="price", ax=ax)
_ = ax.set_title("Aspect Ratio vs Price", fontsize=15)
_ = ax.set_ylabel("Price")
_ = ax.set_xlabel("Aspect Ratio")
_ = ax.set_xlim((0.995, 1.05))

As we can see, we have no good separation when we look at aspect ratio alone as an engineered feature. However, there may be opportunity here to combine the aspect ratio with another signal such as the carat weight. Any weak signal we have in the aspect ratio could be amplified. If we look at carat weight alone for a moment, we see the following:

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 10))

_ = sns.scatterplot(data=train_copy, x="carat", y="price", hue="price", ax=ax)
_ = ax.set_title("Carat vs Price", fontsize=15)
_ = ax.set_ylabel("Price")
_ = ax.set_xlabel("Carat")

As we can see, carat weight has a large impact on price. What we're looking for are ways to modify carat so that we can create good separation of different price points. If we combine it with aspect ratio - which likely has a weak separation signal - we may be able to tease apart more information. 

In [ ]:
train_copy = train.copy()
train_copy["y"] = train_copy["y"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["x"] = train_copy["x"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["new_y"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y <= row.x else row.x, axis=1)
train_copy["new_x"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y >= row.x else row.x, axis=1)
train_copy["aspect_ratio_carat"] = (train_copy["new_x"] / train_copy["new_y"]) / train_copy["carat"] ** 3

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(20, 10))

_ = sns.scatterplot(data=train_copy, x="aspect_ratio_carat", y="price", hue="price", ax=ax)
_ = ax.set_title("Aspect Ratio / Carats ** 3 vs Price", fontsize=15)
_ = ax.set_ylabel("Price")
_ = ax.set_xlabel("Aspect Ratio / Carats ** 3")
_ = ax.set_xlim((0, 20))

### Key Observations About X and Y Values

* There is a linear relationship between price and X and Y size:
    * Price increases as X and Y values increase
* The aspect ratio of a gem appears to have two distinct peaks
    * However, this is due to the fact that we aren't standardizing the way we are looking at gems.
    * We may want to consider changing our view of them such that we always have X be the larger dimension of width and length.
* Aspect ratio as a feature alone is unlikely to help separate price ranges.
    * If we combine it with carat weight, we may be able to add signal.

# 3 - Models

Now that we have a basic understanding of our features, we can start to build models to take advantage of our domain knowledge and engineered features.

# 3.1 - LightGBM Regressor

Let's see how a Vanilla LightGBM regressor does.

In [ ]:
from lightgbm import LGBMRegressor
from lightgbm import early_stopping, log_evaluation
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

features = [
    'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z'
]

train_copy = train.copy()

for feature in ["cut", "color", "clarity"]:
    le = LabelEncoder()
    train_copy[feature] = le.fit_transform(train_copy[feature])
    
train_copy["price"] = train_copy["price"].astype(np.float32)
    
n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.zeros((train.shape[0],))
rmse_scores = []

for fold, (train_index, test_index) in enumerate(skf.split(train_copy, train_copy["price"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(train_copy.iloc[train_index]), pd.DataFrame(train_copy.iloc[test_index])
    y_train, y_valid = train_copy["price"].iloc[train_index], train_copy["price"].iloc[test_index]
    
    x_train_features = pd.DataFrame(x_train[features])
    x_valid_features = pd.DataFrame(x_valid[features])

    model = LGBMRegressor(
        random_state=2023,
        objective="rmse",
        n_jobs=-1,
        n_estimators=5000,
        verbose=-1,    
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(5000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    train_oof_preds[test_index] = oof_preds
    local_rmse = mean_squared_error(y_valid, oof_preds, squared=False)
    rmse_scores.append(local_rmse)
    print(": RMSE = {}".format(local_rmse))
    
rmse_vanilla = mean_squared_error(train["price"], train_oof_preds, squared=False)
print("--> Overall results for out of fold predictions")
print(": RMSE = {}".format(rmse_vanilla))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

data = pd.DataFrame({"Fold": [x + 1 for x in range(n_folds)], "RMSE": rmse_scores})
_ = sns.lineplot(x="Fold", y="RMSE", data=data, ax=ax)
_ = ax.set_title("RMSE per Fold", fontsize=15)
_ = ax.set_ylabel("RMSE")
_ = ax.set_xlabel("Fold #")

# 3.2 - Helmert Encoding

Whenever we have categorical values that appear in the model, we can consider different encoding schemes. In the previous model, we used label encoding to encode the categorical features. We can also look at using Helmert encoding. 

In [ ]:
from category_encoders import HelmertEncoder

features = [
    'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z'
]

train_copy = train.copy()

cat_features = ["cut", "color", "clarity"]
for feature in cat_features:
    encoder = HelmertEncoder()
    df = encoder.fit_transform(train_copy[feature], train_copy["price"])
    new_columns = [x for x in df.columns if x.startswith(feature)]
    for column in new_columns:
        train_copy[column] = df[column]
    features.remove(feature)
    features.extend(new_columns)
    
train_copy["price"] = train_copy["price"].astype(np.float32)
    
n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.zeros((train.shape[0],))
rmse_scores = []

for fold, (train_index, test_index) in enumerate(skf.split(train_copy, train_copy["price"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(train_copy.iloc[train_index]), pd.DataFrame(train_copy.iloc[test_index])
    y_train, y_valid = train_copy["price"].iloc[train_index], train_copy["price"].iloc[test_index]
    
    x_train_features = pd.DataFrame(x_train[features])
    x_valid_features = pd.DataFrame(x_valid[features])

    model = LGBMRegressor(
        random_state=2023,
        objective="rmse",
        n_jobs=-1,
        n_estimators=5000,
        verbose=-1,    
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(5000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    train_oof_preds[test_index] = oof_preds
    local_rmse = mean_squared_error(y_valid, oof_preds, squared=False)
    rmse_scores.append(local_rmse)
    print(": RMSE = {}".format(local_rmse))
    
rmse_he = mean_squared_error(train["price"], train_oof_preds, squared=False)
print("--> Overall results for out of fold predictions")
print(": RMSE = {}".format(rmse_he))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

data = pd.DataFrame({"Fold": [x + 1 for x in range(n_folds)], "RMSE": rmse_scores})
_ = sns.lineplot(x="Fold", y="RMSE", data=data, ax=ax)
_ = ax.set_title("RMSE per Fold", fontsize=15)
_ = ax.set_ylabel("RMSE")
_ = ax.set_xlabel("Fold #")

# 3.3 - One-Hot Encoding

We should also look at using the default one-hot encoding.

In [ ]:
from category_encoders import OneHotEncoder

features = [
    'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z'
]

train_copy = train.copy()

cat_features = ["cut", "color", "clarity"]
for feature in cat_features:
    encoder = OneHotEncoder()
    df = encoder.fit_transform(train_copy[feature], train_copy["price"])
    new_columns = [x for x in df.columns if x.startswith(feature)]
    for column in new_columns:
        train_copy[column] = df[column]
    features.remove(feature)
    features.extend(new_columns)
    
train_copy["price"] = train_copy["price"].astype(np.float32)
    
n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.zeros((train.shape[0],))
rmse_scores = []

for fold, (train_index, test_index) in enumerate(skf.split(train_copy, train_copy["price"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(train_copy.iloc[train_index]), pd.DataFrame(train_copy.iloc[test_index])
    y_train, y_valid = train_copy["price"].iloc[train_index], train_copy["price"].iloc[test_index]
    
    x_train_features = pd.DataFrame(x_train[features])
    x_valid_features = pd.DataFrame(x_valid[features])

    model = LGBMRegressor(
        random_state=2023,
        objective="rmse",
        n_jobs=-1,
        n_estimators=5000,
        verbose=-1,    
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(5000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    train_oof_preds[test_index] = oof_preds
    local_rmse = mean_squared_error(y_valid, oof_preds, squared=False)
    rmse_scores.append(local_rmse)
    print(": RMSE = {}".format(local_rmse))
    
rmse_ohe = mean_squared_error(train["price"], train_oof_preds, squared=False)
print("--> Overall results for out of fold predictions")
print(": RMSE = {}".format(rmse_ohe))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

data = pd.DataFrame({"Fold": [x + 1 for x in range(n_folds)], "RMSE": rmse_scores})
_ = sns.lineplot(x="Fold", y="RMSE", data=data, ax=ax)
_ = ax.set_title("RMSE per Fold", fontsize=15)
_ = ax.set_ylabel("RMSE")
_ = ax.set_xlabel("Fold #")

# 3.4 - Clarity Feature

We saw from above that the clarity of a gem can be ordered based on it's string description. We can make use of that ordering in our classifier, instead of treating it as an unordered categorical.

In [ ]:
features = [
    'carat', 'cut', 'color', 'clarity_scaled', 'depth', 'table', 'x', 'y', 'z'
]

train_copy = train.copy()

for feature in ["cut", "color"]:
    le = LabelEncoder()
    train_copy[feature] = le.fit_transform(train_copy[feature])
    
train_copy["price"] = train_copy["price"].astype(np.float32)
    
n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.zeros((train.shape[0],))
rmse_scores = []

for fold, (train_index, test_index) in enumerate(skf.split(train_copy, train_copy["price"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(train_copy.iloc[train_index]), pd.DataFrame(train_copy.iloc[test_index])
    y_train, y_valid = train_copy["price"].iloc[train_index], train_copy["price"].iloc[test_index]
    
    x_train_features = pd.DataFrame(x_train[features])
    x_valid_features = pd.DataFrame(x_valid[features])

    model = LGBMRegressor(
        random_state=2023,
        objective="rmse",
        n_jobs=-1,
        n_estimators=5000,
        verbose=-1,    
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(5000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    train_oof_preds[test_index] = oof_preds
    local_rmse = mean_squared_error(y_valid, oof_preds, squared=False)
    rmse_scores.append(local_rmse)
    print(": RMSE = {}".format(local_rmse))
    
rmse_clarity = mean_squared_error(train["price"], train_oof_preds, squared=False)
print("--> Overall results for out of fold predictions")
print(": RMSE = {}".format(rmse_clarity))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

data = pd.DataFrame({"Fold": [x + 1 for x in range(n_folds)], "RMSE": rmse_scores})
_ = sns.lineplot(x="Fold", y="RMSE", data=data, ax=ax)
_ = ax.set_title("RMSE per Fold", fontsize=15)
_ = ax.set_ylabel("RMSE")
_ = ax.set_xlabel("Fold #")

# 3.5 - Color Feature

We saw from above that the color of a gem can be ordered based on it's string description. We can make use of that ordering in our classifier, instead of treating it as an unordered categorical.

In [ ]:
features = [
    'carat', 'cut', 'color_scaled', 'clarity', 'depth', 'table', 'x', 'y', 'z'
]

train_copy = train.copy()

train_copy["color_scaled"] = train_copy["color"].apply(
    lambda x: 9 if x == "D" else 8 if x == "E" else 7 if x == "F" else 6 if x == "G" else 5 if x == "H" else 4 if x == "I" else 3 if x == "J" else 2
)
train_copy["color_scaled"] = train_copy["color_scaled"].astype(np.float32)
    
for feature in ["clarity", "cut"]:
    le = LabelEncoder()
    train_copy[feature] = le.fit_transform(train_copy[feature])
    
train_copy["price"] = train_copy["price"].astype(np.float32)
    
n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.zeros((train.shape[0],))
rmse_scores = []

for fold, (train_index, test_index) in enumerate(skf.split(train_copy, train_copy["price"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(train_copy.iloc[train_index]), pd.DataFrame(train_copy.iloc[test_index])
    y_train, y_valid = train_copy["price"].iloc[train_index], train_copy["price"].iloc[test_index]
    
    x_train_features = pd.DataFrame(x_train[features])
    x_valid_features = pd.DataFrame(x_valid[features])

    model = LGBMRegressor(
        random_state=2023,
        objective="rmse",
        n_jobs=-1,
        n_estimators=5000,
        verbose=-1,    
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(5000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    train_oof_preds[test_index] = oof_preds
    local_rmse = mean_squared_error(y_valid, oof_preds, squared=False)
    rmse_scores.append(local_rmse)
    print(": RMSE = {}".format(local_rmse))
    
rmse_color = mean_squared_error(train["price"], train_oof_preds, squared=False)
print("--> Overall results for out of fold predictions")
print(": RMSE = {}".format(rmse_color))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

data = pd.DataFrame({"Fold": [x + 1 for x in range(n_folds)], "RMSE": rmse_scores})
_ = sns.lineplot(x="Fold", y="RMSE", data=data, ax=ax)
_ = ax.set_title("RMSE per Fold", fontsize=15)
_ = ax.set_ylabel("RMSE")
_ = ax.set_xlabel("Fold #")

# 3.6 - Cut Feature

We saw from above that the cut of a gem can be ordered based on it's string description. We can make use of that ordering in our classifier, instead of treating it as an unordered categorical.

In [ ]:
features = [
    'carat', 'cut_scaled', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z'
]

train_copy = train.copy()

train_copy["cut_scaled"] = train_copy["cut"].apply(
    lambda x: 5 if x == "Ideal" else 4 if x == "Premium" else 3 if x == "Very Good" else 2 if x == "Good" else 1
)
train_copy["cut_scaled"] = train_copy["cut_scaled"].astype(np.float32)
    
for feature in ["clarity", "color"]:
    le = LabelEncoder()
    train_copy[feature] = le.fit_transform(train_copy[feature])
    
train_copy["price"] = train_copy["price"].astype(np.float32)
    
n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.zeros((train.shape[0],))
rmse_scores = []

for fold, (train_index, test_index) in enumerate(skf.split(train_copy, train_copy["price"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(train_copy.iloc[train_index]), pd.DataFrame(train_copy.iloc[test_index])
    y_train, y_valid = train_copy["price"].iloc[train_index], train_copy["price"].iloc[test_index]
    
    x_train_features = pd.DataFrame(x_train[features])
    x_valid_features = pd.DataFrame(x_valid[features])

    model = LGBMRegressor(
        random_state=2023,
        objective="rmse",
        n_jobs=-1,
        n_estimators=5000,
        verbose=-1,    
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(5000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    train_oof_preds[test_index] = oof_preds
    local_rmse = mean_squared_error(y_valid, oof_preds, squared=False)
    rmse_scores.append(local_rmse)
    print(": RMSE = {}".format(local_rmse))
    
rmse_cut = mean_squared_error(train["price"], train_oof_preds, squared=False)
print("--> Overall results for out of fold predictions")
print(": RMSE = {}".format(rmse_cut))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

data = pd.DataFrame({"Fold": [x + 1 for x in range(n_folds)], "RMSE": rmse_scores})
_ = sns.lineplot(x="Fold", y="RMSE", data=data, ax=ax)
_ = ax.set_title("RMSE per Fold", fontsize=15)
_ = ax.set_ylabel("RMSE")
_ = ax.set_xlabel("Fold #")

# 3.7 - Aspect Ratio

From our feature exploration, we also saw that there may be a relationship between aspect ratio and price, specifically potential clustering occurring. Let's check the aspect ratio feature.

In [ ]:
features = [
    'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z', 'aspect_ratio_carat'
]

train_copy = train.copy()
train_copy["y"] = train_copy["y"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["x"] = train_copy["x"].apply(lambda x: 0.1 if x == 0 else x)
train_copy["new_y"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y <= row.x else row.x, axis=1)
train_copy["new_x"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y >= row.x else row.x, axis=1)
train_copy["aspect_ratio_carat"] = (train_copy["new_x"] / train_copy["new_y"]) / train_copy["carat"] ** 3
    
for feature in ["clarity", "color", "cut"]:
    le = LabelEncoder()
    train_copy[feature] = le.fit_transform(train_copy[feature])
    
train_copy["price"] = train_copy["price"].astype(np.float32)
    
n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.zeros((train.shape[0],))
rmse_scores = []

for fold, (train_index, test_index) in enumerate(skf.split(train_copy, train_copy["price"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(train_copy.iloc[train_index]), pd.DataFrame(train_copy.iloc[test_index])
    y_train, y_valid = train_copy["price"].iloc[train_index], train_copy["price"].iloc[test_index]
    
    x_train_features = pd.DataFrame(x_train[features])
    x_valid_features = pd.DataFrame(x_valid[features])

    model = LGBMRegressor(
        random_state=2023,
        objective="rmse",
        n_jobs=-1,
        n_estimators=5000,
        verbose=-1,    
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(5000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    train_oof_preds[test_index] = oof_preds
    local_rmse = mean_squared_error(y_valid, oof_preds, squared=False)
    rmse_scores.append(local_rmse)
    print(": RMSE = {}".format(local_rmse))
    
rmse_ar = mean_squared_error(train["price"], train_oof_preds, squared=False)
print("--> Overall results for out of fold predictions")
print(": RMSE = {}".format(rmse_ar))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

data = pd.DataFrame({"Fold": [x + 1 for x in range(n_folds)], "RMSE": rmse_scores})
_ = sns.lineplot(x="Fold", y="RMSE", data=data, ax=ax)
_ = ax.set_title("RMSE per Fold", fontsize=15)
_ = ax.set_ylabel("RMSE")
_ = ax.set_xlabel("Fold #")

# 3.8 - Standardized View

In order to ensure we are comparing gems the same way, we should standardize the way we look at them. In this case, we should always make sure that we look at a gem such that the X dimension is always the larger of width and length.

In [ ]:
train_copy = train.copy()
train_copy["new_y"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y <= row.x else row.x, axis=1)
train_copy["new_x"] = train_copy[["x", "y"]].apply(lambda row: row.y if row.y >= row.x else row.x, axis=1)

features = [
    'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'new_x', 'new_y', 'z'
]

for feature in ["clarity", "color", "cut"]:
    le = LabelEncoder()
    train_copy[feature] = le.fit_transform(train_copy[feature])
    
train_copy["price"] = train_copy["price"].astype(np.float32)
    
n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.zeros((train.shape[0],))
rmse_scores = []

for fold, (train_index, test_index) in enumerate(skf.split(train_copy, train_copy["price"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(train_copy.iloc[train_index]), pd.DataFrame(train_copy.iloc[test_index])
    y_train, y_valid = train_copy["price"].iloc[train_index], train_copy["price"].iloc[test_index]
    
    x_train_features = pd.DataFrame(x_train[features])
    x_valid_features = pd.DataFrame(x_valid[features])

    model = LGBMRegressor(
        random_state=2023,
        objective="rmse",
        n_jobs=-1,
        n_estimators=5000,
        verbose=-1,    
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(5000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    train_oof_preds[test_index] = oof_preds
    local_rmse = mean_squared_error(y_valid, oof_preds, squared=False)
    rmse_scores.append(local_rmse)
    print(": RMSE = {}".format(local_rmse))
    
rmse_xy_fe = mean_squared_error(train["price"], train_oof_preds, squared=False)
print("--> Overall results for out of fold predictions")
print(": RMSE = {}".format(rmse_xy_fe))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

data = pd.DataFrame({"Fold": [x + 1 for x in range(n_folds)], "RMSE": rmse_scores})
_ = sns.lineplot(x="Fold", y="RMSE", data=data, ax=ax)
_ = ax.set_title("RMSE per Fold", fontsize=15)
_ = ax.set_ylabel("RMSE")
_ = ax.set_xlabel("Fold #")

# 3.9 - Original Data

One thing we can look at is adding in the original data. If we do so, we need to make sure we evaluate the model performance based only on the competition data, so we know how well we are doing. We'll compare this to the original label encoded vanilla model.

In [ ]:
features = [
    'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z'
]

train_copy = train.copy()
original = pd.read_csv("../input/gemstone-price-prediction/cubic_zirconia.csv")

train_copy["origin"] = 0
original["origin"] = 1

for feature in ["cut", "color", "clarity"]:
    le = LabelEncoder()
    train_copy[feature] = le.fit_transform(train_copy[feature])
    original[feature] = le.transform(original[feature])
    
train_copy["price"] = train_copy["price"].astype(np.float32)

n_folds = 5
skf = KFold(n_splits=n_folds, random_state=2023, shuffle=True)
train_oof_preds = np.array([])
train_oof_truth = np.array([])
rmse_scores = []

chunk_size = int(len(original) / n_folds)
original_chunks = [original[i:i+chunk_size] for i in range(0, len(original), chunk_size)]

combined = pd.concat([combined, original]).reset_index(drop=True)

for fold, (train_index, test_index) in enumerate(skf.split(train_copy, train_copy["price"])):
    print("-------> Fold {} <--------".format(fold + 1))
    x_train, x_valid = pd.DataFrame(train_copy.iloc[train_index]), pd.DataFrame(train_copy.iloc[test_index])
    y_train, y_valid = train_copy["price"].iloc[train_index], train_copy["price"].iloc[test_index]
    

    x_train_features = pd.DataFrame(x_train)

    x_train_features = pd.concat([x_train_features, original_chunks[fold]]).reset_index(drop=True)
    y_train = y_train.append(original_chunks[fold]["price"])
    
    x_valid_features = pd.DataFrame(x_valid)

    model = LGBMRegressor(
        random_state=2023,
        objective="rmse",
        n_jobs=-1,
        n_estimators=5000,
        verbose=-1,    
    )
    model.fit(
        x_train_features[features], 
        y_train,
        eval_set=[(x_valid_features[features], y_valid)],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(5000),
        ]
    )
    oof_preds = model.predict(x_valid_features[features])
    train_oof_preds = np.append(train_oof_preds, oof_preds)
    train_oof_truth = np.append(train_oof_truth, y_valid)
    local_rmse = mean_squared_error(y_valid, oof_preds, squared=False)
    rmse_scores.append(local_rmse)
    print(": RMSE = {}".format(local_rmse))
    
rmse_extra = mean_squared_error(train_oof_truth, train_oof_preds, squared=False)
print("--> Overall results for out of fold predictions")
print(": RMSE = {}".format(rmse_extra))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

data = pd.DataFrame({"Fold": [x + 1 for x in range(n_folds)], "RMSE": rmse_scores})
_ = sns.lineplot(x="Fold", y="RMSE", data=data, ax=ax)
_ = ax.set_title("RMSE per Fold", fontsize=15)
_ = ax.set_ylabel("RMSE")
_ = ax.set_xlabel("Fold #")

# 3.x - Model Comparison

Let's take a look at how the models perform.

In [ ]:
bar, ax = plt.subplots(figsize=(20, 10))
ax = sns.barplot(
    x=["Baseline", "Hilbert Encode", "One-Hot Encode", "Clarity FE", "Color FE", "Cut FE", "Aspect Ratio FE", "Standard View", "Extra Data"],
    y=[
        rmse_vanilla,
        rmse_he,
        rmse_ohe,
        rmse_clarity,
        rmse_color,
        rmse_cut,
        rmse_ar,
        rmse_xy_fe,
        rmse_extra,
    ]
)
_ = ax.axhline(y=577.6949, color='r', linestyle='--')
_ = ax.set_title("Root Mean Squared Error (Lower is Better)", fontsize=15)
_ = ax.set_xlabel("")
_ = ax.set_ylabel("RMSE")
_ = ax.set_ylim([576, 579])
for p in ax.patches:
    height = p.get_height()
    ax.text(
        x=p.get_x()+(p.get_width()/2),
        y=height,
        s="{:.4f}".format(height),
        ha="center"
    )

# 4 - Conclusions

There are several interesting conclusions we can draw:

* The dataset is small in terms of disk space and memory usage, allowing a number of machine learning techniques to be used without worry of memory pressure.
* There are no null values in the training and testing sets, however, some rows with zero values exist and can be partially imputed using other features.
* Both the training and testing set look very similar to one another.
    * Adversarial validation suggests that we can also use the original dataset, however, there are null values that need to be imputed in that dataset before we use it.
* Several features can be engineered from the existing data:
    * Cut, clarity, and color can be replaced with ordered numeric values instead.
    * Aspect ratio can be combined with carat value to provide lift to the model.
    * The original data can be combined with the competition data to provide additional lift.